# FlashTile: Flash Attention Demo

_Outputs are cleared in the committed notebook so the file stays source-only. Run the cells to reproduce current results._

This notebook is a compact walkthrough of the Flash Attention implementations in this repo.

**Contents**
- How Flash Attention achieves O(N) memory complexity
- Memory savings vs naive attention
- Different attention variants (V1, V2, GQA)
- Custom Triton kernel performance

## 1. Setup and Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# FlashTile imports
from flashtile import (
    NaiveAttention,
    FlashAttentionV1,
    FlashAttentionV2,
    GroupedQueryAttention,
    MultiQueryAttention,
    check_installation,
)

# Check system
info = check_installation()
print(f"PyTorch: {info['torch_version']}")
print(f"CUDA Available: {info['cuda_available']}")
if info['cuda_available']:
    print(f"CUDA Version: {info['cuda_version']}")
print(f"Triton Available: {info['triton_available']}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

## 2. Basic Usage: Flash Attention V2

Flash Attention V2 is the recommended implementation for most use cases.
It includes causal block skipping for ~50% compute savings on decoder models.

In [ ]:
# Configuration
embed_dim = 512
num_heads = 8
seq_len = 1024
batch_size = 2

# Create model
model = FlashAttentionV2(
    embed_dim=embed_dim,
    num_heads=num_heads,
    causal=True,  # For decoder-style models
).to(device)

# Create input
x = torch.randn(batch_size, seq_len, embed_dim, device=device)

# Forward pass
output, attention_weights = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights: {attention_weights}")  # None - memory efficient!

## 3. Memory Comparison: Naive vs Flash

Let's measure actual memory usage across different sequence lengths.

In [ ]:
def measure_memory(model, x):
    """Measure peak memory usage."""
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
    
    with torch.no_grad():
        _ = model(x)
    
    if device == "cuda":
        return torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
    return 0.0


# Test different sequence lengths
seq_lengths = [256, 512, 1024, 2048]
naive_memory = []
flash_memory = []

for seq_len in seq_lengths:
    x = torch.randn(batch_size, seq_len, embed_dim, device=device)
    
    # Naive
    try:
        naive = NaiveAttention(embed_dim, num_heads).to(device).eval()
        mem = measure_memory(naive, x)
        naive_memory.append(mem)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            naive_memory.append(float('inf'))
            if device == "cuda":
                torch.cuda.empty_cache()
    
    # Flash V2
    flash = FlashAttentionV2(embed_dim, num_heads).to(device).eval()
    mem = measure_memory(flash, x)
    flash_memory.append(mem)
    
    print(f"Seq len {seq_len}: Naive={naive_memory[-1]:.1f}MB, Flash={flash_memory[-1]:.1f}MB")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, naive_memory, 'o-', label='Naive (O(N²))', linewidth=2, markersize=8)
plt.plot(seq_lengths, flash_memory, 's-', label='Flash V2 (O(N))', linewidth=2, markersize=8)
plt.xlabel('Sequence Length', fontsize=12)
plt.ylabel('Memory (MB)', fontsize=12)
plt.title('Memory Usage: Naive vs Flash Attention', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

## 4. Causal Block Skipping Efficiency

Flash Attention V2 skips blocks that would be entirely masked in causal attention.
This saves ~50% compute for decoder models.

In [ ]:
model = FlashAttentionV2(embed_dim=512, num_heads=8, causal=True)

for seq_len in [1024, 2048, 4096]:
    efficiency = model.get_causal_efficiency(seq_len)
    print(f"\nSequence length: {seq_len}")
    print(f"  Total block pairs: {efficiency['total_blocks']}")
    print(f"  Computed: {efficiency['computed_blocks']}")
    print(f"  Skipped: {efficiency['skipped_blocks']}")
    print(f"  Compute savings: {efficiency['compute_savings_percent']:.1f}%")

## 5. Grouped Query Attention (GQA)

GQA reduces KV cache memory by sharing key/value heads across query heads.
Used in LLaMA 2, Mistral, and other production models.

In [ ]:
embed_dim = 512
num_heads = 8
batch_size = 2
seq_len = 2048

# Standard Multi-Head Attention (MHA)
mha = FlashAttentionV2(embed_dim, num_heads, causal=True)
mha_mem = mha.get_memory_usage(batch_size, seq_len)

# Grouped Query Attention (4x reduction)
gqa = GroupedQueryAttention(embed_dim, num_heads, num_kv_heads=2, causal=True)
gqa_mem = gqa.get_memory_usage(batch_size, seq_len)

# Multi-Query Attention (8x reduction - maximum)
mqa = MultiQueryAttention(embed_dim, num_heads, causal=True)
mqa_mem = mqa.get_memory_usage(batch_size, seq_len)

print("KV Cache Memory Comparison:")
print(f"  MHA:  {mha_mem['total_mb']:.1f} MB")
print(f"  GQA:  {gqa_mem['total_mb']:.1f} MB ({gqa_mem['kv_savings_ratio']:.0f}x reduction)")
print(f"  MQA:  {mqa_mem['total_mb']:.1f} MB ({mqa_mem['kv_savings_ratio']:.0f}x reduction)")

## 6. Correctness Verification

Flash Attention should match naive attention within floating-point tolerance.

In [ ]:
def sync_weights(target, source):
    """Copy weights from source to target."""
    target.qkv_proj.weight.data = source.qkv_proj.weight.data.clone()
    target.qkv_proj.bias.data = source.qkv_proj.bias.data.clone()
    target.out_proj.weight.data = source.out_proj.weight.data.clone()
    target.out_proj.bias.data = source.out_proj.bias.data.clone()


# Create models
naive = NaiveAttention(embed_dim, num_heads).to(device).eval()
flash_v1 = FlashAttentionV1(embed_dim, num_heads).to(device).eval()
flash_v2 = FlashAttentionV2(embed_dim, num_heads).to(device).eval()

# Sync weights
sync_weights(flash_v1, naive)
sync_weights(flash_v2, naive)

# Test input
x = torch.randn(2, 512, embed_dim, device=device)

# Forward pass
with torch.no_grad():
    naive_out, _ = naive(x)
    v1_out, _ = flash_v1(x)
    v2_out, _ = flash_v2(x)

# Compare
v1_diff = (naive_out - v1_out).abs().max().item()
v2_diff = (naive_out - v2_out).abs().max().item()

print("Numerical Correctness Check:")
print(f"  Max difference (Flash V1 vs Naive): {v1_diff:.2e}")
print(f"  Max difference (Flash V2 vs Naive): {v2_diff:.2e}")
print(f"  Pass (threshold: 1e-3)") if max(v1_diff, v2_diff) < 1e-3 else print(f"  Fail")

## 7. Backward Pass (Training)

FlashTile includes memory-efficient backward passes that recompute
attention during backprop instead of storing the N² matrix.

In [ ]:
# Create model and input
model = FlashAttentionV2(embed_dim=256, num_heads=4, causal=True).to(device)
x = torch.randn(2, 128, 256, device=device, requires_grad=True)

# Forward pass
output, _ = model(x)
loss = output.sum()

# Backward pass
loss.backward()

print(f"Loss: {loss.item():.4f}")
print(f"Input gradient shape: {x.grad.shape}")
print(f"Gradient contains NaN: {torch.isnan(x.grad).any().item()}")
print(f"Backward pass successful!")

## 8. Triton Kernel (Optional)

If Triton is installed, you can use the optimized GPU kernel.

In [ ]:
if info['triton_available']:
    from flashtile import TritonFlashAttention
    
    # Create Triton model (forward-only, must use eval + no_grad)
    triton_model = TritonFlashAttention(embed_dim=512, num_heads=8, causal=True).to(device).eval()
    
    # Test
    x = torch.randn(2, 1024, 512, device=device)
    with torch.no_grad():
        output, _ = triton_model(x)
    
    print(f"Triton kernel output shape: {output.shape}")
    print(f"Triton kernel working!")
else:
    print("Triton not installed. Install with: pip install triton")

## 9. Summary

**Summary:**

1. **Memory Efficiency**: Flash Attention achieves O(N) memory vs O(N²) naive
2. **Exact Computation**: Outputs match standard attention up to floating-point tolerance
3. **Training Support**: Full backward pass with O(N) memory
4. **Causal Optimization**: V2 skips ~50% of compute for decoder models
5. **Inference Optimization**: GQA/MQA reduce KV cache by 4-8x

**Typical use:**
- **Flash V2**: Default for decoder models (GPT-style)
- **Flash V1**: When you need the original algorithm
- **GQA**: Inference optimization (LLaMA 2-style)
- **Triton**: Maximum performance on compatible GPUs